In [1]:
import os
import shutil
import random
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
# Change this to where you uploaded the unzipped Kaggle dataset
SOURCE_DATA_DIR = '/content/drive/MyDrive/INFO813_Project/animal_dataset'
# This is where the YOLO-ready data will go
OUTPUT_DIR = '/content/drive/MyDrive/INFO813_Project/yolo_ready_data'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- 1. DATA ANALYSIS & VISUALIZATION ---
classes = [d for d in os.listdir(SOURCE_DATA_DIR) if os.path.isdir(os.path.join(SOURCE_DATA_DIR, d))]
class_counts = {}

for cls in classes:
    num_images = len(os.listdir(os.path.join(SOURCE_DATA_DIR, cls)))
    class_counts[cls] = num_images

# Plotting the visualization for your report
plt.figure(figsize=(20, 8))
plt.bar(class_counts.keys(), class_counts.values(), color='skyblue')
plt.xticks(rotation=90, fontsize=8)
plt.title('Distribution of Images per Animal Class')
plt.xlabel('Animal Classes')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/INFO813_Project/class_distribution.png')
plt.show()

print(f"Total Classes: {len(classes)}")
print(f"Total Images: {sum(class_counts.values())}")
print("Visualization saved! Use this image in your report.")



In [ ]:
# --- 2. DATA TRANSFORMATION (TRAIN/VAL SPLIT) ---
train_dir = os.path.join(OUTPUT_DIR, 'train')
val_dir = os.path.join(OUTPUT_DIR, 'val')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

split_ratio = 0.8 # 80% for training, 20% for validation

for cls in classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)

    images = os.listdir(os.path.join(SOURCE_DATA_DIR, cls))
    random.shuffle(images) # Shuffle to ensure random distribution

    split_index = int(len(images) * split_ratio)
    train_images = images[:split_index]
    val_images = images[split_index:]

    # Copy files
    for img in train_images:
        shutil.copy(os.path.join(SOURCE_DATA_DIR, cls, img), os.path.join(train_dir, cls, img))
    for img in val_images:
        shutil.copy(os.path.join(SOURCE_DATA_DIR, cls, img), os.path.join(val_dir, cls, img))

print("✅ Data successfully split into Train and Val folders!")

In [3]:
# 1. Install the YOLO library
!pip install ultralytics

# 2. Import YOLO
from ultralytics import YOLO

# 3. Load the pre-trained Nano Classification Model
# We use 'n' (nano) because it trains much faster on Colab while still giving good results
model = YOLO('yolo11n-cls.pt')

# 4. Train the Model!
# IMPORTANT: The 'data' path must match the OUTPUT_DIR from Step 1
results = model.train(
    data='/content/drive/MyDrive/INFO813_Project/yolo_ready_data',
    epochs=20,          # 20 epochs is a good start. If accuracy is low, we can increase this later.
    imgsz=224,          # YOLO classification uses 224x224 image sizes
    batch=32,           # Processes 32 images at a time
    project='/content/drive/MyDrive/INFO813_Project/training_results',
    name='animal_classifier'
)

# 5. Evaluate the model on the Validation set
metrics = model.val()
print("--- TRAINING COMPLETE ---")
print(f"Top-1 Accuracy: {metrics.top1 * 100:.2f}%")
print(f"Top-5 Accuracy: {metrics.top5 * 100:.2f}%")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/INFO813_Proje